In [ ]:
import numpy as np

In [2]:
def build_tokens(sentence):
    tokens = ["[SOS]"] + sentence.split() + ["[EOS]"]
    return tokens

def build_embeddings(tokens):
    embeddings = {}
    for token in tokens:
        token_lower = token.lower()
        if token_lower in ["bank"]:
            embeddings[token] = np.array([0.8, 0.7, 0.6, 0.2])
        elif token_lower in ["money", "cash", "deposit"]:
            embeddings[token] = np.array([0.9, 0.8, 0.7, 0.1])
        elif token_lower in ["river", "stream"]:
            embeddings[token] = np.array([0.9, 0.2, 0.1, 0.8])
        elif token_lower in ["going", "deposited", "is", "hit"]:
            embeddings[token] = np.array([0.2, 0.3, 0.1, 0.0])
        else:
            embeddings[token] = np.random.rand(4) * 0.2
    return embeddings

In [3]:
def compute_raw_scores(query_token, tokens, embeddings):
    raw_scores = []
    q = embeddings[query_token]
    for token in tokens:
        k = embeddings[token]
        score = np.dot(q, k)
        raw_scores.append(score)
    return np.array(raw_scores, dtype=float)

def scale_scores(raw_scores, d_k=64):
    return raw_scores / np.sqrt(d_k)

def softmax(scores):
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / np.sum(exp_scores)

In [4]:
def run_attention_experiment(sentence, query_token):
    print("="*70)
    print(f"Input sentence:\n{sentence}\n")
    
    tokens = build_tokens(sentence)
    embeddings = build_embeddings(tokens)
    
    print(f"Tokens:\n{tokens}\n")
    print(f"Query token: '{query_token}'\n")
    
    raw_scores = compute_raw_scores(query_token, tokens, embeddings)
    print("Step 1: Raw similarity scores (Q·K):")
    for t, s in zip(tokens, raw_scores):
        print(f"{t:10s} -> {s:.4f}")
    print()
    
    scaled_scores = scale_scores(raw_scores)
    print("Step 2: Scaled scores:")
    for t, s in zip(tokens, scaled_scores):
        print(f"{t:10s} -> {s:.4f}")
    print()
    
    attention_weights = softmax(scaled_scores)
    print("Step 3: Attention weights (softmax):")
    for t, s in zip(tokens, attention_weights):
        print(f"{t:10s} -> {s:.4f}")
    print("\n")
    
    most_influential = tokens[np.argmax(attention_weights)]
    print("\nConclusion:")
    print(
        f"The word '{most_influential}' contributes the most to the meaning of '{query_token}'.\n"
    )

In [5]:
sentence_1 = "He deposited money in the bank"
query_token_1 = "bank"
run_attention_experiment(sentence_1, query_token_1)

Input sentence:
He deposited money in the bank

Tokens:
['[SOS]', 'He', 'deposited', 'money', 'in', 'the', 'bank', '[EOS]']

Query token: 'bank'

Step 1: Raw similarity scores (Q·K):
[SOS]      -> 0.1541
He         -> 0.3708
deposited  -> 0.4300
money      -> 1.7200
in         -> 0.2538
the        -> 0.2326
bank       -> 1.5300
[EOS]      -> 0.2172

Step 2: Scaled scores:
[SOS]      -> 0.0193
He         -> 0.0463
deposited  -> 0.0538
money      -> 0.2150
in         -> 0.0317
the        -> 0.0291
bank       -> 0.1912
[EOS]      -> 0.0271

Step 3: Attention weights (softmax):
[SOS]      -> 0.1177
He         -> 0.1209
deposited  -> 0.1218
money      -> 0.1431
in         -> 0.1192
the        -> 0.1189
bank       -> 0.1398
[EOS]      -> 0.1186



Conclusion:
The word 'money' contributes the most to the meaning of 'bank'.



In [6]:
sentence_2 = "He is going to the river's bank"
query_token_2 = "bank"
run_attention_experiment(sentence_2, query_token_2)

Input sentence:
He is going to the river's bank

Tokens:
['[SOS]', 'He', 'is', 'going', 'to', 'the', "river's", 'bank', '[EOS]']

Query token: 'bank'

Step 1: Raw similarity scores (Q·K):
[SOS]      -> 0.3405
He         -> 0.2278
is         -> 0.4300
going      -> 0.4300
to         -> 0.2899
the        -> 0.3125
river's    -> 0.1827
bank       -> 1.5300
[EOS]      -> 0.1594

Step 2: Scaled scores:
[SOS]      -> 0.0426
He         -> 0.0285
is         -> 0.0538
going      -> 0.0538
to         -> 0.0362
the        -> 0.0391
river's    -> 0.0228
bank       -> 0.1912
[EOS]      -> 0.0199

Step 3: Attention weights (softmax):
[SOS]      -> 0.1097
He         -> 0.1081
is         -> 0.1109
going      -> 0.1109
to         -> 0.1090
the        -> 0.1093
river's    -> 0.1075
bank       -> 0.1273
[EOS]      -> 0.1072



Conclusion:
The word 'bank' contributes the most to the meaning of 'bank'.



In [7]:
sentence_3 = "He hit the ball with a bat"
query_token_3 = "bat"
run_attention_experiment(sentence_3, query_token_3)

Input sentence:
He hit the ball with a bat

Tokens:
['[SOS]', 'He', 'hit', 'the', 'ball', 'with', 'a', 'bat', '[EOS]']

Query token: 'bat'

Step 1: Raw similarity scores (Q·K):
[SOS]      -> 0.0548
He         -> 0.0314
hit        -> 0.0826
the        -> 0.0780
ball       -> 0.0565
with       -> 0.0436
a          -> 0.0459
bat        -> 0.0691
[EOS]      -> 0.0361

Step 2: Scaled scores:
[SOS]      -> 0.0068
He         -> 0.0039
hit        -> 0.0103
the        -> 0.0098
ball       -> 0.0071
with       -> 0.0054
a          -> 0.0057
bat        -> 0.0086
[EOS]      -> 0.0045

Step 3: Attention weights (softmax):
[SOS]      -> 0.1111
He         -> 0.1108
hit        -> 0.1115
the        -> 0.1114
ball       -> 0.1111
with       -> 0.1109
a          -> 0.1110
bat        -> 0.1113
[EOS]      -> 0.1108



Conclusion:
The word 'hit' contributes the most to the meaning of 'bat'.

